# Desafio de Análise de Dados Industriais

## Objetivo

Realizar uma análise completa dos dados de produção industrial com o objetivo de identificar gargalos operacionais, avaliar o desempenho produtivo e desenvolver uma previsão da produção real com apoio de técnicas de machine learning.

---

## Contexto

Nos últimos meses, a operação da fábrica apresentou diferenças recorrentes entre a produção planejada e a produção realizada, além de registros relevantes de paradas de máquina e variações na taxa de defeitos. Diante desse cenário, torna-se necessário investigar os dados disponíveis para compreender os fatores que estão impactando o desempenho da produção.

---

## Escopo da Análise

O trabalho deverá contemplar as seguintes etapas:

### 1. Análise Exploratória dos Dados
- Examinar o comportamento geral da base de dados
- Avaliar o desempenho por máquina, produto e turno
- Identificar padrões, tendências e possíveis anomalias

### 2. Identificação de Gargalos Operacionais
- Verificar quais máquinas apresentam maior tempo de parada
- Identificar quais produtos possuem menor eficiência produtiva
- Avaliar quais turnos apresentam melhor e pior desempenho

### 3. Análise de Relações entre Variáveis
Investigar a relação entre:
- tempo de parada de máquina
- taxa de defeitos
- quantidade de trabalhadores no turno
- produção real

### 4. Criação de Indicador de Eficiência
Desenvolver pelo menos uma métrica de apoio à análise.

**Sugestão de métrica:**

`Eficiência produtiva = produção real / produção planejada`

### 5. Modelagem Preditiva
- Construir um modelo de machine learning para prever a variável **produção real**
- Utilizar como base as variáveis disponíveis no conjunto de dados
- Avaliar o desempenho do modelo com métricas adequadas

### 6. Apresentação dos Resultados
- Organizar a análise com gráficos e interpretações objetivas
- Destacar os principais gargalos encontrados
- Apresentar insights relevantes e possíveis ações de melhoria

### 7. Diferencial
- Propor uma ideia inicial de automação para execução recorrente da análise e geração de relatórios

---

## Entrega Esperada

Ao final do projeto, espera-se uma análise clara e estruturada, contendo:

- exploração dos dados
- identificação de gargalos
- indicador de eficiência
- previsão da produção real
- conclusões e recomendações

In [48]:
# ============================================================
# ETAPA 1 - ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)
# Objetivos:
# - Examinar o comportamento geral da base de dados
# - Avaliar o desempenho por máquina, produto e turno
# - Identificar padrões, tendências e possíveis anomalias
# ============================================================

# 1. Importação das bibliotecas
import pandas as pd
import plotly.express as px

# 2. Carregamento da base de dados
df = pd.read_excel("factory_production_dataset.xlsx")

# 3. Visualização inicial
print("Prévia da base de dados:")
display(df.head())

# 4. Informações gerais da base
print("Dimensões da base de dados:", df.shape)
print("\nInformações gerais:")
df.info()

# 5. Estatísticas descritivas
print("\nEstatísticas descritivas das variáveis numéricas:")
display(df.describe())

# 6. Verificação de valores nulos
print("\nQuantidade de valores nulos por coluna:")
display(df.isnull().sum())

# 7. Verificação de registros duplicados
print("\nQuantidade de linhas duplicadas:", df.duplicated().sum())

# 8. Conversão da coluna de data
df["date"] = pd.to_datetime(df["date"])

# 9. Criação do indicador de eficiência produtiva
df["efficiency_percent"] = (df["actual_production"] / df["planned_production"]) * 100

print("\nPrévia da base com o indicador de eficiência:")
display(df.head())

# ============================================================
# ANÁLISE DE DESEMPENHO POR TURNO
# ============================================================
shift_performance = (
    df.groupby("shift", as_index=False)["efficiency_percent"]
    .mean()
    .sort_values(by="efficiency_percent", ascending=False)
)

fig = px.bar(
    shift_performance,
    x="shift",
    y="efficiency_percent",
    title="Eficiência Média por Turno",
    labels={
        "shift": "Turno",
        "efficiency_percent": "Eficiência Média (%)"
    },
    text_auto=".2f"
)
fig.show()

# ============================================================
# ANÁLISE DE DESEMPENHO POR PRODUTO
# ============================================================
product_performance = (
    df.groupby("product_id", as_index=False)["efficiency_percent"]
    .mean()
    .sort_values(by="efficiency_percent", ascending=False)
)

fig = px.bar(
    product_performance,
    x="product_id",
    y="efficiency_percent",
    title="Eficiência Média por Produto",
    labels={
        "product_id": "Produto",
        "efficiency_percent": "Eficiência Média (%)"
    },
    text_auto=".2f"
)
fig.show()

# ============================================================
# ANÁLISE DE DESEMPENHO POR MÁQUINA
# ============================================================
machine_performance = (
    df.groupby("machine_id", as_index=False)["efficiency_percent"]
    .mean()
    .sort_values(by="efficiency_percent", ascending=False)
)

fig = px.bar(
    machine_performance,
    x="machine_id",
    y="efficiency_percent",
    title="Eficiência Média por Máquina",
    labels={
        "machine_id": "Máquina",
        "efficiency_percent": "Eficiência Média (%)"
    },
    text_auto=".2f"
)
fig.show()

# ============================================================
# DISTRIBUIÇÃO DA EFICIÊNCIA PRODUTIVA
# ============================================================
fig = px.histogram(
    df,
    x="efficiency_percent",
    nbins=30,
    title="Distribuição da Eficiência Produtiva",
    labels={"efficiency_percent": "Eficiência (%)"}
)
fig.show()

# ============================================================
# ANÁLISE TEMPORAL DA PRODUÇÃO REAL
# ============================================================
production_over_time = (
    df.groupby("date", as_index=False)["actual_production"]
    .mean()
)

fig = px.line(
    production_over_time,
    x="date",
    y="actual_production",
    title="Produção Média ao Longo do Tempo",
    labels={
        "date": "Data",
        "actual_production": "Produção Média Real"
    }
)
fig.show()

# ============================================================
# TEMPO MÉDIO DE PARADA POR MÁQUINA
# ============================================================
machine_downtime = (
    df.groupby("machine_id", as_index=False)["machine_downtime_minutes"]
    .mean()
    .sort_values(by="machine_downtime_minutes", ascending=False)
)

fig = px.bar(
    machine_downtime,
    x="machine_id",
    y="machine_downtime_minutes",
    title="Tempo Médio de Parada por Máquina",
    labels={
        "machine_id": "Máquina",
        "machine_downtime_minutes": "Tempo Médio de Parada (min)"
    },
    text_auto=".2f"
)
fig.show()

# ============================================================
# TAXA MÉDIA DE DEFEITOS POR PRODUTO
# ============================================================
product_defect_rate = (
    df.groupby("product_id", as_index=False)["defect_rate"]
    .mean()
    .sort_values(by="defect_rate", ascending=False)
)

fig = px.bar(
    product_defect_rate,
    x="product_id",
    y="defect_rate",
    title="Taxa Média de Defeitos por Produto",
    labels={
        "product_id": "Produto",
        "defect_rate": "Taxa Média de Defeitos"
    },
    text_auto=".3f"
)
fig.show()

# ============================================================
# RELAÇÃO ENTRE TEMPO DE PARADA E PRODUÇÃO REAL
# ============================================================
fig = px.scatter(
    df,
    x="machine_downtime_minutes",
    y="actual_production",
    color="machine_id",
    title="Relação entre Tempo de Parada e Produção Real",
    labels={
        "machine_downtime_minutes": "Tempo de Parada (min)",
        "actual_production": "Produção Real"
    }
)
fig.show()

Prévia da base de dados:


,date,product_id,machine_id,shift,planned_production,actual_production,machine_downtime_minutes,defect_rate,workers_on_shift
0,2025-01-01,A,M1,night,131,118,13,0.021,5
1,2025-01-01,A,M2,night,103,91,22,0.061,4
2,2025-01-01,A,M3,morning,112,81,33,0.040,3
3,2025-01-01,A,M4,night,138,120,22,0.078,5
4,2025-01-01,B,M1,night,143,124,47,0.058,3


Dimensões da base de dados: (2880, 9)

Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2880 entries, 0 to 2879
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      2880 non-null   datetime64[ns]
 1   product_id                2880 non-null   object        
 2   machine_id                2880 non-null   object        
 3   shift                     2880 non-null   object        
 4   planned_production        2880 non-null   int64         
 5   actual_production         2880 non-null   int64         
 6   machine_downtime_minutes  2880 non-null   int64         
 7   defect_rate               2880 non-null   float64       
 8   workers_on_shift          2880 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(4), object(3)
memory usage: 202.6+ KB

Estatísticas descritivas das variáveis numéricas:


,date,planned_production,actual_production,machine_downtime_minutes,defect_rate,workers_on_shift
count,2880,2880.000000,2880.000000,2880.000000,2880.000000,2880.000000
mean,2025-03-31 12:00:00,119.269444,104.315278,29.320139,0.045001,5.018750
min,2025-01-01 00:00:00,80.000000,54.000000,0.000000,0.010000,3.000000
25%,2025-02-14 18:00:00,99.000000,87.000000,19.000000,0.027000,4.000000
50%,2025-03-31 12:00:00,119.000000,103.000000,29.000000,0.045000,5.000000
75%,2025-05-15 06:00:00,139.000000,122.000000,39.000000,0.062000,6.000000
max,2025-06-29 00:00:00,159.000000,159.000000,82.000000,0.080000,7.000000
std,NaN,23.025945,22.330460,14.897506,0.020298,1.415562



Quantidade de valores nulos por coluna:


date                        0
product_id                  0
machine_id                  0
shift                       0
planned_production          0
actual_production           0
machine_downtime_minutes    0
defect_rate                 0
workers_on_shift            0
dtype: int64


Quantidade de linhas duplicadas: 0

Prévia da base com o indicador de eficiência:


,date,product_id,machine_id,shift,planned_production,actual_production,machine_downtime_minutes,defect_rate,workers_on_shift,efficiency_percent
0,2025-01-01,A,M1,night,131,118,13,0.021,5,90.076336
1,2025-01-01,A,M2,night,103,91,22,0.061,4,88.349515
2,2025-01-01,A,M3,morning,112,81,33,0.040,3,72.321429
3,2025-01-01,A,M4,night,138,120,22,0.078,5,86.956522
4,2025-01-01,B,M1,night,143,124,47,0.058,3,86.713287


In [49]:
# ============================================================
# ETAPA 2 - IDENTIFICAÇÃO DE GARGALOS OPERACIONAIS
# Objetivos:
# - Verificar quais máquinas apresentam maior tempo de parada
# - Identificar quais produtos possuem menor eficiência produtiva
# - Avaliar quais turnos apresentam melhor e pior desempenho
# ============================================================

# ============================================================
# 1. MÁQUINAS COM MAIOR TEMPO DE PARADA
# ============================================================
machine_downtime_summary = (
    df.groupby("machine_id", as_index=False)["machine_downtime_minutes"]
    .sum()
)

machine_downtime_summary["machine_downtime_hours"] = (
    machine_downtime_summary["machine_downtime_minutes"] / 60
)

machine_downtime_summary = machine_downtime_summary.sort_values(
    by="machine_downtime_hours",
    ascending=False
)

fig = px.bar(
    machine_downtime_summary,
    x="machine_id",
    y="machine_downtime_hours",
    title="Máquinas com Maior Tempo Total de Parada",
    labels={
        "machine_id": "Máquina",
        "machine_downtime_hours": "Tempo de Parada (horas)"
    },
    text_auto=".2f"
)
fig.show()

print(
    f"A máquina com maior tempo de parada é a {machine_downtime_summary.iloc[0]['machine_id']}, "
    f"com um total de {machine_downtime_summary.iloc[0]['machine_downtime_hours']:.2f} horas."
)

# ============================================================
# 2. PRODUTOS COM MENOR EFICIÊNCIA PRODUTIVA
# ============================================================
product_efficiency_summary = (
    df.groupby("product_id", as_index=False)["efficiency_percent"]
    .mean()
    .sort_values(by="efficiency_percent", ascending=True)
)

fig = px.bar(
    product_efficiency_summary,
    x="product_id",
    y="efficiency_percent",
    title="Produtos com Menor Eficiência Produtiva",
    labels={
        "product_id": "Produto",
        "efficiency_percent": "Eficiência Média (%)"
    },
    text_auto=".2f"
)
fig.show()

print(
    f"O produto com menor eficiência produtiva é o produto {product_efficiency_summary.iloc[0]['product_id']}, "
    f"com eficiência média de {product_efficiency_summary.iloc[0]['efficiency_percent']:.2f}%."
)

# ============================================================
# 3. DESEMPENHO POR TURNO
# ============================================================
shift_performance_summary = (
    df.groupby("shift", as_index=False)["efficiency_percent"]
    .mean()
    .sort_values(by="efficiency_percent", ascending=False)
)

fig = px.bar(
    shift_performance_summary,
    x="shift",
    y="efficiency_percent",
    title="Desempenho Médio por Turno",
    labels={
        "shift": "Turno",
        "efficiency_percent": "Eficiência Média (%)"
    },
    text_auto=".2f"
)
fig.show()

print(
    f"O turno com melhor desempenho é o {shift_performance_summary.iloc[0]['shift']}, "
    f"com eficiência média de {shift_performance_summary.iloc[0]['efficiency_percent']:.2f}%."
)

print(
    f"O turno com pior desempenho é o {shift_performance_summary.iloc[-1]['shift']}, "
    f"com eficiência média de {shift_performance_summary.iloc[-1]['efficiency_percent']:.2f}%."
)

A máquina com maior tempo de parada é a M3, com um total de 356.17 horas.


O produto com menor eficiência produtiva é o produto A, com eficiência média de 87.19%.


O turno com melhor desempenho é o afternoon, com eficiência média de 87.58%.
O turno com pior desempenho é o morning, com eficiência média de 87.32%.


In [50]:
# ============================================================
# ETAPA 3 - ANÁLISE DE RELAÇÕES ENTRE VARIÁVEIS
# Objetivo:
# Investigar a relação entre variáveis operacionais e a produção real
# ============================================================

# Seleção das variáveis numéricas mais relevantes
correlation_data = df[
    [
        "planned_production",
        "actual_production",
        "machine_downtime_minutes",
        "defect_rate",
        "workers_on_shift",
        "efficiency_percent"
    ]
]

# ============================================================
# 1. MATRIZ DE CORRELAÇÃO
# ============================================================
correlation_matrix = correlation_data.corr(numeric_only=True)

display(correlation_matrix)

fig = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    aspect="auto",
    title="Matriz de Correlação entre Variáveis Operacionais"
)
fig.show()

# ============================================================
# 2. RELAÇÃO ENTRE TEMPO DE PARADA E PRODUÇÃO REAL
# ============================================================
fig = px.scatter(
    df,
    x="machine_downtime_minutes",
    y="actual_production",
    color="machine_id",
    title="Relação entre Tempo de Parada e Produção Real",
    labels={
        "machine_downtime_minutes": "Tempo de Parada (min)",
        "actual_production": "Produção Real"
    }
)
fig.show()

# ============================================================
# 3. RELAÇÃO ENTRE TAXA DE DEFEITOS E PRODUÇÃO REAL
# ============================================================
fig = px.scatter(
    df,
    x="defect_rate",
    y="actual_production",
    color="product_id",
    title="Relação entre Taxa de Defeitos e Produção Real",
    labels={
        "defect_rate": "Taxa de Defeitos",
        "actual_production": "Produção Real"
    }
)
fig.show()

# ============================================================
# 4. RELAÇÃO ENTRE QUANTIDADE DE TRABALHADORES E PRODUÇÃO REAL
# ============================================================
fig = px.scatter(
    df,
    x="workers_on_shift",
    y="actual_production",
    color="shift",
    title="Relação entre Quantidade de Trabalhadores e Produção Real",
    labels={
        "workers_on_shift": "Quantidade de Trabalhadores",
        "actual_production": "Produção Real"
    }
)
fig.show()

# ============================================================
# 5. RELAÇÃO ENTRE PRODUÇÃO PLANEJADA E PRODUÇÃO REAL
# ============================================================
fig = px.scatter(
    df,
    x="planned_production",
    y="actual_production",
    color="product_id",
    title="Relação entre Produção Planejada e Produção Real",
    labels={
        "planned_production": "Produção Planejada",
        "actual_production": "Produção Real"
    }
)
fig.show()

,planned_production,actual_production,machine_downtime_minutes,defect_rate,workers_on_shift,efficiency_percent
planned_production,1.000000,0.909740,0.001728,-0.005384,0.017044,0.020569
actual_production,0.909740,1.000000,-0.002693,0.004936,0.003780,0.425971
machine_downtime_minutes,0.001728,-0.002693,1.000000,-0.003372,-0.004650,-0.010603
defect_rate,-0.005384,0.004936,-0.003372,1.000000,-0.030149,0.021905
workers_on_shift,0.017044,0.003780,-0.004650,-0.030149,1.000000,-0.022755
efficiency_percent,0.020569,0.425971,-0.010603,0.021905,-0.022755,1.000000


In [51]:
# ============================================================
# ETAPA 4 - CRIAÇÃO DE INDICADOR DE EFICIÊNCIA
# Objetivo:
# Criar uma métrica para avaliar o desempenho produtivo
# ============================================================

# Criação do indicador de eficiência produtiva
df["efficiency_percent"] = (df["actual_production"] / df["planned_production"]) * 100

# Visualização inicial do indicador
display(df[["planned_production", "actual_production", "efficiency_percent"]].head())

# Estatísticas do indicador
print("Estatísticas do indicador de eficiência:")
display(df["efficiency_percent"].describe())

# Gráfico da distribuição da eficiência
fig = px.histogram(
    df,
    x="efficiency_percent",
    nbins=30,
    title="Distribuição da Eficiência Produtiva",
    labels={"efficiency_percent": "Eficiência (%)"}
)
fig.show()

print(
    f"A eficiência média geral da produção é de {df['efficiency_percent'].mean():.2f}%."
)

,planned_production,actual_production,efficiency_percent
0,131,118,90.076336
1,103,91,88.349515
2,112,81,72.321429
3,138,120,86.956522
4,143,124,86.713287


Estatísticas do indicador de eficiência:


count    2880.000000
mean       87.431313
std         7.695925
min        59.139785
25%        82.352941
50%        87.671233
75%        93.083554
max       100.000000
Name: efficiency_percent, dtype: float64

A eficiência média geral da produção é de 87.43%.


In [52]:
# ============================================================
# ETAPA 5 - MODELAGEM PREDITIVA
# Objetivo:
# Construir um modelo para prever a produção real com base em
# variáveis operacionais
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Seleção de variáveis preditoras e variável alvo
X = df[[
    "planned_production",
    "machine_downtime_minutes",
    "defect_rate",
    "workers_on_shift"
]]

y = df["actual_production"]

# Separação em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Treinamento do modelo
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# Previsões
y_pred = model.predict(X_test)

# Avaliação do modelo
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("Avaliação do modelo preditivo:")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

# Comparação entre valores reais e previstos
resultados = pd.DataFrame({
    "actual_production_real": y_test.values,
    "actual_production_predita": y_pred
})

display(resultados.head())

# Gráfico de comparação entre valores reais e previstos
fig = px.scatter(
    resultados,
    x="actual_production_real",
    y="actual_production_predita",
    title="Produção Real vs Produção Predita",
    labels={
        "actual_production_real": "Produção Real",
        "actual_production_predita": "Produção Predita"
    }
)
fig.show()

Avaliação do modelo preditivo:
MAE: 8.20
MSE: 106.81
RMSE: 10.34
R²: 0.7786


,actual_production_real,actual_production_predita
0,88,96.72
1,118,116.36
2,92,86.26
3,140,119.19
4,131,133.84


In [53]:
# ============================================================
# ETAPA 6 - APRESENTAÇÃO DOS RESULTADOS
# Objetivo:
# Consolidar os principais insights obtidos ao longo da análise
# ============================================================

# Máquina com maior tempo de parada
top_machine = machine_downtime_summary.iloc[0]

# Produto com menor eficiência
worst_product = product_efficiency_summary.iloc[0]

# Melhor e pior turno
best_shift = shift_performance_summary.iloc[0]
worst_shift = shift_performance_summary.iloc[-1]

print("========== PRINCIPAIS RESULTADOS ==========")
print(
    f"1. Máquina com maior tempo de parada: {top_machine['machine_id']} "
    f"({top_machine['machine_downtime_hours']:.2f} horas)."
)

print(
    f"2. Produto com menor eficiência produtiva: {worst_product['product_id']} "
    f"({worst_product['efficiency_percent']:.2f}%)."
)

print(
    f"3. Turno com melhor desempenho: {best_shift['shift']} "
    f"({best_shift['efficiency_percent']:.2f}%)."
)

print(
    f"4. Turno com pior desempenho: {worst_shift['shift']} "
    f"({worst_shift['efficiency_percent']:.2f}%)."
)

print(
    f"5. Eficiência média geral da operação: {df['efficiency_percent'].mean():.2f}%."
)

print(
    f"6. Desempenho do modelo preditivo - R²: {r2:.4f}, RMSE: {rmse:.2f}."
)

========== PRINCIPAIS RESULTADOS ==========
1. Máquina com maior tempo de parada: M3 (356.17 horas).
2. Produto com menor eficiência produtiva: A (87.19%).
3. Turno com melhor desempenho: afternoon (87.58%).
4. Turno com pior desempenho: morning (87.32%).
5. Eficiência média geral da operação: 87.43%.
6. Desempenho do modelo preditivo - R²: 0.7786, RMSE: 10.34.


In [54]:
# ============================================================
# ETAPA 7 - CONCLUSÕES E RECOMENDAÇÕES
# Objetivo:
# Sintetizar os achados e propor ações de melhoria
# ============================================================

print("========== CONCLUSÕES ==========")
print(
    "A análise permitiu identificar diferenças relevantes de desempenho entre máquinas, "
    "produtos e turnos, indicando a existência de gargalos operacionais na produção."
)

print(
    "O tempo de parada das máquinas se mostrou um fator importante para a redução da produção real, "
    "assim como a variação na taxa de defeitos e na eficiência produtiva."
)

print(
    "O modelo preditivo apresentou capacidade de estimar a produção real com base nas variáveis operacionais, "
    "demonstrando potencial de apoio à tomada de decisão."
)

print("\n========== RECOMENDAÇÕES ==========")
print(
    "- Investigar as causas do alto tempo de parada da máquina com pior desempenho."
)
print(
    "- Avaliar o processo produtivo do produto com menor eficiência para identificar perdas."
)
print(
    "- Revisar práticas operacionais do turno com pior desempenho."
)
print(
    "- Monitorar continuamente os indicadores de eficiência, downtime e defeitos."
)
print(
    "- Evoluir a solução para relatórios automatizados e acompanhamento recorrente."
)

========== CONCLUSÕES ==========
A análise permitiu identificar diferenças relevantes de desempenho entre máquinas, produtos e turnos, indicando a existência de gargalos operacionais na produção.
O tempo de parada das máquinas se mostrou um fator importante para a redução da produção real, assim como a variação na taxa de defeitos e na eficiência produtiva.
O modelo preditivo apresentou capacidade de estimar a produção real com base nas variáveis operacionais, demonstrando potencial de apoio à tomada de decisão.

========== RECOMENDAÇÕES ==========
- Investigar as causas do alto tempo de parada da máquina com pior desempenho.
- Avaliar o processo produtivo do produto com menor eficiência para identificar perdas.
- Revisar práticas operacionais do turno com pior desempenho.
- Monitorar continuamente os indicadores de eficiência, downtime e defeitos.
- Evoluir a solução para relatórios automatizados e acompanhamento recorrente.
